In [1]:
# ======= Variables =========

# Fill in the variables below and run cell first
# Note: Use forward slashes / instead of backslashes \ in file paths

DATASET_DIR = "../dataset"
OUTPUT_DIR = f"{DATASET_DIR}/original_sheets"

In [3]:
# Read the dim_patient, dim_physician and fact_txn DataFrames

from pathlib import Path

import pandas as pd

output_path = Path(OUTPUT_DIR)

dim_patient = pd.read_csv(output_path / "dim_patient.csv")
dim_physician = pd.read_csv(output_path / "dim_physician.csv")
fact_txn = pd.read_csv(output_path / "fact_txn.csv")

model_table_original = pd.read_csv(output_path / "model_table.csv")

print(f"dim_patient: rows={dim_patient.shape[0]}, columns={dim_patient.shape[1]}")
print(f"dim_physician: rows={dim_physician.shape[0]}, columns={dim_physician.shape[1]}")
print(f"fact_txn: rows={fact_txn.shape[0]}, columns={fact_txn.shape[1]}")
print(
    f"model_table_original: rows={model_table_original.shape[0]}, columns={model_table_original.shape[1]}"
)

dim_patient: rows=4020, columns=3
dim_physician: rows=25343, columns=5
fact_txn: rows=115274, columns=7
model_table_original: rows=9, columns=4


In [6]:
# Check whether there is a single patient with more than one physician in the fact_txn table
patient_physician_counts = fact_txn.groupby("PATIENT_ID")["PHYSICIAN_ID"].nunique()
patients_with_multiple_physicians = patient_physician_counts[patient_physician_counts > 1]
patients_with_multiple_physicians

PATIENT_ID
9        2
11       2
14       3
24       2
28       5
        ..
4016    26
4017    53
4018    25
4019    10
4020    10
Name: PHYSICIAN_ID, Length: 3264, dtype: int64

In [13]:
# Transform fact_txn into a patient-level transaction structure
import sys
from importlib import reload
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import ml_antiviral_diagnosis.de as de

reload(de)

patient_transactions_df = de.transform_fact_txn_to_patient_transactions(fact_txn)

print(
    f"patient_transactions_df: rows={patient_transactions_df.shape[0]}, columns={patient_transactions_df.shape[1]}"
)
display(patient_transactions_df.head())
print("Sample transactions for the first patient:")
patient_transactions_df.iloc[0]["transactions_by_type"]

patient_transactions_df: rows=4020, columns=2


,patient_id,transactions_by_type
0,1,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
1,2,"{'SYMPTOMS': [{'txn_dt': '2022-06-22', 'physic..."
2,3,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
3,4,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."
4,5,"{'SYMPTOMS': [], 'CONDITIONS': [{'txn_dt': '20..."


Sample transactions for the first patient:


{'SYMPTOMS': [],
 'CONDITIONS': [{'txn_dt': '2022-06-11',
   'physician_id': 24633,
   'txn_location_type': 'EMERGENCY ROOM - HOSPITAL',
   'insurance_type': 'COMMERCIAL',
   'txn_desc': 'disease_x'}],
 'CONTRAINDICATIONS': [],
 'TREATMENTS': []}